**Setup**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, importlib
REPO = "/content/anomaly-mvtec"
if not os.path.exists(REPO):
    !git clone https://github.com/maha-mrrk/anomaly-mvtec.git {REPO}
else:
    !git -C {REPO} pull
%cd {REPO}
if REPO not in sys.path:
    sys.path.append(REPO)
importlib.invalidate_caches()          # prend en compte les nouveaux fichiers après un pull
!pip install -q -r requirements.txt

from src.config import *
set_seed(42)
prepare_data()
print(device, DATA_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Already up to date.
/content/anomaly-mvtec
Données prêtes : ['carpet', 'screw', 'cable']
cuda /content/mvtec


**Imports**

In [2]:
import torch, numpy as np
from src.data import *
from src.scoring import *
from src.metrics import *

In [10]:
!wc -c /content/anomaly-mvtec/src/scoring.py /content/anomaly-mvtec/src/metrics.py

1157 /content/anomaly-mvtec/src/scoring.py
1929 /content/anomaly-mvtec/src/metrics.py
3086 total


**Tests sur les 3 catégories**

In [3]:
for cat in CATEGORIES:
    tr, val = list_train_val(DATA_ROOT, cat)
    test_ds = MVTecDataset(list_test(DATA_ROOT, cat), img_size=128)
    train_ds = MVTecDataset(tr, img_size=128)

    # labels et masques (la carte renvoyée ici ne sert pas)
    _, labels, masks = predict_maps(lambda x: torch.zeros(len(x), 128, 128),
                                    test_ds, device=device)

    # 1) Oracle : AUROC attendu = 1.0
    print(cat, "| Oracle    :", evaluate(labels, masks, masks.copy(), sigma=0))

    # 2) Aléatoire : AUROC attendu ≈ 0.5
    rand = np.random.rand(*masks.shape).astype("float32")
    print(cat, "| Aléatoire :", evaluate(labels, masks, rand, sigma=0))

    # 3) Baseline « image moyenne »
    mean_img = torch.stack([train_ds[i][0] for i in range(len(train_ds))]).mean(0).to(device)
    maps, labels, masks = predict_maps(lambda x: (x - mean_img).abs().mean(dim=1),
                                       test_ds, device=device)
    res = evaluate(labels, masks, maps, with_aupro=True)
    print(cat, "| Baseline  :", res, "\n")
    log_result(RESULTS_CSV, model="baseline_mean_img", category=cat, **res)

carpet | Oracle    : {'img_auroc': np.float64(1.0), 'pix_auroc': np.float64(1.0)}
carpet | Aléatoire : {'img_auroc': np.float64(0.4295746388443018), 'pix_auroc': np.float64(0.49991476792620704)}
carpet | Baseline  : {'img_auroc': np.float64(0.3952648475120385), 'pix_auroc': np.float64(0.7147357763646416), 'aupro': np.float64(0.4604497389414164)} 

screw | Oracle    : {'img_auroc': np.float64(1.0), 'pix_auroc': np.float64(1.0)}
screw | Aléatoire : {'img_auroc': np.float64(0.4919040787046526), 'pix_auroc': np.float64(0.5025586533251324)}
screw | Baseline  : {'img_auroc': np.float64(0.23037507686001232), 'pix_auroc': np.float64(0.87769820445759), 'aupro': np.float64(0.6280495500374976)} 

cable | Oracle    : {'img_auroc': np.float64(1.0), 'pix_auroc': np.float64(1.0)}
cable | Aléatoire : {'img_auroc': np.float64(0.4491191904047976), 'pix_auroc': np.float64(0.5004874997825509)}
cable | Baseline  : {'img_auroc': np.float64(0.6353073463268365), 'pix_auroc': np.float64(0.9014680731593547), 'a

**Tableau des résultats**

In [4]:
import pandas as pd
pd.read_csv(RESULTS_CSV)

,model,category,img_auroc,pix_auroc,aupro
0,baseline_mean_img,carpet,0.395265,0.714736,0.460450
1,baseline_mean_img,screw,0.230375,0.877698,0.628050
2,baseline_mean_img,cable,0.635307,0.901468,0.532381
